In [13]:
from collections.abc import AsyncIterable

from pydantic_ai import Agent, RunUsage, AgentStreamEvent, RunContext
from pydantic_ai.capabilities import ProcessEventStream, Instrumentation, WebSearch
from pydantic_ai.messages import FunctionToolCallEvent
from pydantic_ai.models.anthropic import AnthropicModel, AnthropicModelSettings
from pydantic_ai.models.instrumented import InstrumentationSettings

import invoke_llm_trace

In [14]:
instructions = """
You're a research agent, your task is to find articles on the internet and 
answer the question from the user. Make sure you include references in the 
answer you generate
"""

In [15]:
async def log_events(ctx: RunContext, events: AsyncIterable[AgentStreamEvent]) -> None:
    async for event in events:
        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            agent_name = ctx.agent.name if ctx.agent and hasattr(ctx.agent, "name") else "search"
            print(f"TOOL CALL ({agent_name}): {tool_name}({args})")

In [ ]:

model = AnthropicModel("claude-opus-5")
model_settings = AnthropicModelSettings(
    anthropic_cache=True,
    anthropic_cache_tool_definitions="1h",
)
model_capabilities = [
    WebSearch(max_uses=4),
    Instrumentation(InstrumentationSettings(version=5)),
    ProcessEventStream(log_events),
]

search_agent = Agent(
    name="search",
    model=model,
    instructions=instructions,
    model_settings=model_settings,
    capabilities=model_capabilities,
)


In [17]:
user_prompt = "How do I do RAG in 2026?"
result = await search_agent.run(user_prompt)

/home/abram/Desktop/Werkstatten/alexey-ai-engineering-buildcamp/.venv/lib/python3.14/site-packages/pydantic_ai/models/anthropic.py:1233: UserWarning: Sampling parameters ['temperature'] are not supported by 'claude-opus-5'. These settings will be ignored.
  self._drop_unsupported_sampling_settings(filtered)


In [18]:
result.output

'# How to Do RAG in 2026\n\nThe short version: **RAG didn\'t die, it grew up.** It\'s no longer "chunk → embed → cosine similarity → stuff into prompt." In 2026 it\'s treated as a *context engineering* discipline — a layered retrieval system where you add complexity only when evaluation proves you need it.\n\nHere\'s the current playbook.\n\n---\n\n## 1. Start with the right mental model\n\nThe consensus advice is refreshingly conservative: the best architecture is the **simplest one that meets your evidence, latency, accuracy, privacy, and maintenance requirements**. Traditional or hybrid RAG is usually the right starting point — you add reranking, graph reasoning, multimodal retrieval, memory, or agents only when evaluation shows a *specific* failure mode ([Turing Post, 20 Advanced RAG Types to Know in 2026](https://www.turingpost.com/p/ragtypes)).\n\nMost "RAG is broken" complaints aren\'t about RAG — they\'re about **naive** RAG: poor chunking, weak ranking, and no thought given to

In [20]:
request, response = result.new_messages()

In [22]:
response.parts

[ThinkingPart(content='', signature='CAIS4QQKwwIIEhgCKkBMgeFLFaJ2SISRKxW9ihLCSz5UL755Nhz8WwsSNz1XfShYYo8kSQpHlxyB/yelNbGJaK/Wfg3frsI3dNE+5FNzMg1jbGF1ZGUtb3B1cy01OAFCCHRoaW5raW5nWiQwMGRmNTJlYi0wYThhLTRhYmMtYjRmOS0xNTJkMWI2MTFmMDWaAQ8KDWNsYXVkZS1vcHVzLTWoAZqIutYGwgE9CAUQABoPAVY2AmyEA083BXQpBCatIAIyBBJxzyU6BNjOiZ1CBKWm6YVKBO/umTpQAloE2QXhVmACaAFwAdoBYAoWcHJlZml4X2JlZm9yZV9ibG9ja192MRIg4cc29k/bWNRGjVM7AODZMoUTYBA3rtgXSEwUidKWfI4iADIgwsATyGqckuIvlm89hhd2j4t6RMkCrcwJe/oeCYr7Th9IAxIMypN0LgitHf4TIu33GgxlnZ8+BECjvNCgFsoiMM6Z3kA84GUwKiaj27hKO2rDhZSOZ0jycT2uqsCYzxCJCA16xeuQGYd0xIpcASLPjCrKAXy+17x5CharG8jB8hQA9rU0vBcydFc3NdJXPu2O9GOzymAi1O77fEPtJ7MhMh6YyJooeUbzEouEH1Uu41vqQLH1KDyBDV5SMHVTu9PLhwTQaJEgqvPIJC6jqO9VKXJUOMjerkKvAh1/DiftwPAtIyQ2S5PAaDb6EfDGvJAiAMQuDlZsPn4Gvid4SYSLOhGyl0EmATXPVfZWUVd8mKx3Djrkh4WhgOaRL3EYuxpa5cWfIsHxQpo7mcJeg//co4CcdTr5aGwHACCa194YAQ==', provider_name='anthropic'),
 TextPart(content="I'll research current RAG best practices for you."),
 NativeToolCallPart(tool_name='code_

In [23]:
result.usage

RunUsage(cache_read_tokens=93957, cache_write_tokens=17940, cost=Decimal('0.4339685'), details={'input_tokens': 18, 'output_tokens': 6991, 'cache_creation_input_tokens': 17940, 'cache_read_input_tokens': 93957, 'thinking_tokens': 775, 'web_search_requests': 10}, input_tokens=111915, output_tokens=6991, requests=1, web_searches=10)